# 01 — Run Pan-human Azimuth on the lab datasets (zero-shot)

For each lab dataset: build a raw-counts AnnData (panhumanpy detects integer counts and applies
CP10k + log1p itself, on all query genes, before subsetting to its 5,055-gene panel; missing panel
genes are zero-filled), run `AzimuthNN` (model v1), map labels to Cell Ontology, and save per-cell
predictions + 128-d embeddings to `results/`.

Notes on outputs:
- `azimuth_broad/medium/fine == 'False'` means panhumanpy's label refinement found no match
  (mostly cells with `full_consistent_hierarchy == False`); `None` means the lineage has no label at
  that resolution. `final_level_labels` is always populated.
- `Unassigned` is PHA's built-in QC label (empty droplet / ambient / doublet-like profile).

In [1]:
import os, sys
from pathlib import Path

# Run from the repo root (found via config.py) so relative paths are stable
_root = Path.cwd()
while not (_root / "config.py").exists():
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

'3'

In [2]:
import time
import numpy as np
import pandas as pd
import panhumanpy as ph

from config import DATASETS, RESULTS_DIR, MODEL_VERSION, load_counts_adata

RESULTS_DIR.mkdir(exist_ok=True)
CL_COLS = ["final_level_labels", "azimuth_broad", "azimuth_medium", "azimuth_fine"]

In [3]:
def run_dataset(name):
    q = load_counts_adata(name)
    t0 = time.time()
    az = ph.AzimuthNN(q, model_version=MODEL_VERSION)
    for col in CL_COLS:
        # panhumanpy 1.0.0 bug: map_to_cell_ontology sorts unmapped labels and crashes on None
        # (lineages with no label at that resolution), so make them an explicit string first
        az.cells_meta[col] = az.cells_meta[col].fillna("None").astype(str)
        az.map_to_cell_ontology(col, include_cl_id=True)
    emb = az.azimuth_embed()
    elapsed = time.time() - t0

    meta = az.cells_meta.copy()
    meta.index.name = "cell_id"
    meta.to_csv(RESULTS_DIR / f"pha_{name}_predictions.csv")
    np.save(RESULTS_DIR / f"pha_{name}_embed.npy", emb)

    panel = set(az.inference_feature_panel)
    overlap = len(panel & set(q.var_names))
    return {
        "dataset": name,
        "n_cells": q.n_obs,
        "n_query_genes": q.n_vars,
        "panel_genes": len(panel),
        "panel_overlap": overlap,
        "panel_overlap_pct": round(100 * overlap / len(panel), 1),
        "unassigned_pct": round(100 * (meta["final_level_labels"] == "Unassigned").mean(), 2),
        "inconsistent_hierarchy_pct": round(100 * (~meta["full_consistent_hierarchy"].astype(bool)).mean(), 2),
        "median_final_conf": round(meta["final_level_confidence"].median(), 3),
        "seconds": round(elapsed, 1),
    }

In [4]:
summary = []
for name in DATASETS:
    print(f"\n===== {name} =====")
    summary.append(run_dataset(name))

summary = pd.DataFrame(summary)
summary.to_csv(RESULTS_DIR / "pha_run_summary.csv", index=False)
summary


===== glioma =====


/home/hugolab/Documents/Cell2Sentence-HugoLab/panhumanpy-HugoLab/.venv/lib/python3.12/site-packages/panhumanpy/ANNotate.py:1402: UserWarning: Ensure that the features metadata is indexed with gene names.
  warnings.warn(


Integer counts detected by panhumanpy.ANNotate : True
Log-normalization performed by panhumanpy.ANNotate: True
To override log-normalization, set normalization_override=False


Query object:
    Total number of cells: 2609
    Total number of features: 27102
    Overlap w. feature reference: 4625 (~91%)

Splitting query data into 1 evaluation batch(es) of up to 
8192 cells.

Interpreting label predictions for consistent granularity at broad, medium, fine level(s).

Running model:


I0000 00:00:1791930467.524690  247425 service.cc:146] XLA service 0x752060034260 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791930467.524721  247425 service.cc:154]   StreamExecutor device (0): Host, Default Version


...

171/195 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

187/195 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


,dataset,n_cells,n_query_genes,panel_genes,panel_overlap,panel_overlap_pct,unassigned_pct,inconsistent_hierarchy_pct,median_final_conf,seconds
0,glioma,2609,27102,5055,4625,91.5,0.00,28.52,0.828,9.8
1,Brain_normal,2773,14814,5055,3963,78.4,0.87,12.41,0.990,10.3
2,Liver_normal,2806,14095,5055,3753,74.2,0.00,5.77,0.997,11.6
3,Lymph_node_normal,2357,13717,5055,3664,72.5,0.00,14.30,0.945,8.4
4,lymphoid,7173,13993,5055,3793,75.0,0.08,11.54,0.938,21.1
5,myeloid,6218,14395,5055,3919,77.5,0.00,11.68,0.957,18.3


## Quick look: top PHA fine label per true label

In [5]:
for name in DATASETS:
    m = pd.read_csv(RESULTS_DIR / f"pha_{name}_predictions.csv", index_col=0)
    top = (m.groupby("true_label")["final_level_labels"]
             .agg(lambda s: ", ".join(f"{k} ({v})" for k, v in s.value_counts().head(3).items())))
    print(f"\n### {name}")
    print(top.to_string())


### glioma
true_label
AC-like            Ciliary body cell (67), Glial cell (52), Fibro...
Astrocyte          Fibrous astrocyte (100), Ciliary body cell (90...
CD4/CD8            CD4 T cell (52), T cell (38), Memory CD4 T cel...
DC                                 Microglia (3), cDC1 (2), cDC2 (1)
Endothelial        Arterial EC (123), Capillary EC (44), Venous E...
MES-like           Ciliary body cell (95), Glial cell (23), Fibro...
NPC-like           Neuron (99), Amacrine cell (41), Neuron of ret...
Neuron             L4 excitatory neuron (27), TSHZ2 L4/5 excitato...
OPC                Oligodendrocyte progenitor cell (150), Splatte...
OPC-like           Glial cell (61), Oligodendrocyte progenitor ce...
Oligodendrocyte    Oligodendrocyte (137), Eye melanocyte (37), Ci...
Pericyte           Pericyte (101), TNC fibroblast (29), Red blood...
TAM-BDM            Microglia (69), Macrophage (52), LYVE1 macroph...
TAM-MG             Microglia (119), Macrophage (51), Tissue resid...

### Brain_